# Credit Agreement Analyzer - Phase 4 Validation
Testing Q&A engine against the real Ribbon Communications credit agreement.

Prerequisites: Ollama running locally with `llama3:8b` loaded.

In [2]:
import contextlib
import time
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker, build_search_text
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))

## Step 0: Rebuild Pipeline (Phase 1-3)
Re-run extraction through retrieval setup.

In [3]:
# Phase 1: Extract, detect, parse, chunk
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)

defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
print(f"Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

Pages: 289 | Sections: 157 | Terms: 391 | Chunks: 718


In [4]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()}")
print(f"Available: {llm.is_available()}")

Provider: claude-sonnet-4-20250514
Available: True


In [5]:
# Phase 3: Retrieval layer
embedder = Embedder()

start = time.time()
embeddings = embedder.embed([build_search_text(c) for c in chunks])
print(f"Embedded {len(chunks)} chunks in {time.time() - start:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_p4")
store.create_collection("ribbon_p4")
store.add_chunks("ribbon_p4", chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)
print("Retrieval layer ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1404.32it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 718 chunks in 25.0s
Retrieval layer ready


## Step 1: Basic Q&A
Test single questions with no conversation history.

In [6]:
qa = QAEngine(retriever=retriever, llm=llm)
print("QAEngine initialized")

QAEngine initialized


In [7]:
def print_response(resp: QAResponse) -> None:
    """Display a QAResponse in a readable format."""
    print(f"Answer:\n{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  Section {s.section_id} - {s.section_title} (pp. {pages})")
        if s.relevant_text_snippet:
            print(f"    Snippet: {s.relevant_text_snippet[:120]}...")
    print(f"\nChunks retrieved: {len(resp.retrieved_chunks)}")
    print(f"LLM time: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used}")
    print("-" * 80)

In [8]:
# Q1: Total revolving commitment (exact dollar amount)
qa.clear_history()
print("Q: What is the total revolving commitment amount?\n")
resp = qa.ask("What is the total revolving commitment amount?", "ribbon_p4")
print_response(resp)

Q: What is the total revolving commitment amount?

Answer:
The total revolving commitment amount is **$35,000,000**.

This is stated in the definition of "Total Revolving Commitments" in Section 1.1, which specifies: "The original amount of the Total Revolving Commitments is $35,000,000."

Key details about this facility:
- The L/C Commitment operates as a sublimit within the Total Revolving Commitments
- The borrower can reduce or terminate the revolving commitments with 3 business days' notice in increments of $1,000,000 (Section 2.10)
- The facility can potentially be increased through incremental facilities, subject to certain conditions and the Available Incremental Amount (Section 2.27)

Confidence: HIGH
Sources (1):
  Section 1.1 - Defined Terms (pp. 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56)
    Snippet: “Total Revolving Commitments

In [9]:
# Q2: Financinal covenants (list them)
qa.clear_history()
print("Q: What financial covenants are included in the agreement?\n")
resp = qa.ask("What financial covenants are included in the agreement?", "ribbon_p4")
print_response(resp)

Q: What financial covenants are included in the agreement?

Answer:
Based on the provided context, the agreement includes **one financial covenant**:

## Maximum Consolidated Net Leverage Ratio (Section 7.1(a))

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following thresholds, tested quarterly starting September 30, 2024:

**Covenant Schedule:**
- **September 2024 through December 2025**: 4.75:1.00
- **March 2026 and thereafter**: 4.00:1.00 (step-down)

The covenant is tested as of the last day of any period of four consecutive trailing fiscal quarters.

## Equity Cure Provision (Section 7.1(b))

If the borrower fails the leverage test, it has an **equity cure** option:
- Cash equity contributions can be made within **10 business days** after financial statements are due
- The contribution is added to Consolidated Adjusted EBITDA for covenant calculation purposes
- **100% of cure proceeds must immediately prepay Term Loans** (Section 7.1(b)(vii

In [10]:
# Q3: Incremental debt capacity (your core use case)
qa.clear_history()
print("Q: How much incremental debt can the borrower incur?\n")
resp = qa.ask("How much incremental debt can the borrower incur?", "ribbon_p4")
print_response(resp)

Q: How much incremental debt can the borrower incur?

Answer:
The borrower can incur incremental debt up to the **Available Incremental Amount**, which is defined as the sum of two components (Section 2.27(a)):

## 1. Fixed Incremental Amount
- **$50 million** or **50% of Consolidated Adjusted EBITDA** (whichever is greater)
- Reduced by any previously incurred Incremental Facilities that relied on this basket

## 2. Ratio Incremental Amount  
This is an unlimited amount, subject to leverage ratio tests that vary by security structure:

- **First lien pari passu**: Consolidated First Lien Net Leverage Ratio ≤ 2.90x
- **Junior lien**: Consolidated Secured Net Leverage Ratio ≤ 3.40x  
- **Unsecured**: Consolidated Net Leverage Ratio ≤ 3.90x

## Key Constraints
- **Revolving vs. Term Mix**: After any revolving commitment increase, total revolving commitments cannot exceed 10% of term loans (Section 2.27(a))
- **Pricing Restriction**: Pari passu incremental facilities cannot have an all-in

In [11]:
# Q4: Restricted payments (complex multi-basket answer)
qa.clear_history()
print("Q: What restricted payments are permitted under the credit agreement?\n")
resp = qa.ask("What restricted payments are permitted under the credit agreement?", "ribbon_p4")
print_response(resp)

Q: What restricted payments are permitted under the credit agreement?

Answer:
Based on Section 7.6, the credit agreement permits several categories of restricted payments, provided no Event of Default has occurred or would result:

## Key Permitted Restricted Payments:

**1. Intercompany Payments (Section 7.6(a))**
- Subsidiaries may make restricted payments to Loan Parties
- Non-Loan Party subsidiaries may make payments to other Group Members or equity holders on pro rata basis

**2. Employee-Related Payments (Section 7.6(b))**
- Purchase of stock/options from directors, officers, employees with tiered limits:
  - $2.5M annually (unused amounts carry forward)
  - Additional $5M if leverage ≤ 2.40:1.00
  - Additional $10M if leverage ≤ 1.90:1.00
- Stock dividends in common stock only
- Employee tax payments for retirement/stock plans

**3. Stock Repurchases (Section 7.6(e), (f), (k))**
- Repurchases funded by concurrent common stock issuance
- Stock option exercise repurchases and tax

In [12]:
# Q5: NOT FOUND test - question about something unlikely to be in the doc
qa.clear_history()
print("Q: What are the details of the borrower's pet insurance policy?\n")
resp = qa.ask("What are the details of the borrower's pet insurance policy?", "ribbon_p4")
print_response(resp)

Q: What are the details of the borrower's pet insurance policy?

Answer:
Based on my review of the provided context, there is no information about the borrower's pet insurance policy. The credit agreement contains general insurance provisions but does not specify details about pet insurance or any other specific types of insurance policies.

The insurance-related provisions I can see are:

**General Insurance Requirements:**
- Section 4.24 requires that all insurance maintained by Loan Parties be in full force and effect with premiums paid
- Section 6.6(b) requires maintaining insurance with financially sound companies "in at least such amounts and against at least such risks (but including in any event public liability, product liability and business interruption) as are usually insured against in the same general area by companies engaged in the same or a similar business"

**Other Insurance References:**
- Section 6.12 mentions title and extended coverage insurance for real property

## Step 2: Conversation Follow-ups
Test that question reformulation works for multi-turn Q&A.

In [13]:
qa.clear_history()

# Turn 1: Establish context
print("Turn 1: What is the term loan facility size?\n")
resp1 = qa.ask("What is the term loan facility size?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up using "it" (requires reformulation)
print("\nTurn 2: What is the interest rate on it?\n")
resp2 = qa.ask("What is the interest rate on it?", "ribbon_p4")
print_response(resp2)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What is the term loan facility size?

Answer:
The term loan facility size is **$350,000,000**.

This is stated in multiple places in the agreement:

1. **Section 2.1** confirms that "The original aggregate principal amount of the Term Commitments on the Closing Date is $350,000,000"

2. The **Preamble** describes the overall credit facilities as "an aggregate principal amount not to exceed $385,000,000, consisting of a term loan facility in the aggregate principal amount of $350,000,000, and a revolving loan facility to the Borrower in an aggregate principal amount of $35,000,000"

The term facility can potentially be increased through incremental facilities under Section 2.27, but the base term loan facility established on the Closing Date is $350 million.

Confidence: HIGH
Sources (2):
  Section 2.1 - Term Commitments (pp. 61)
    Snippet: 2.1 Term Commitments. Subject to the terms and conditions hereof, each Term Lender severally agrees to make a Term Loan ...
  Section 1.1 

In [14]:
qa.clear_history()

# Turn 1: Ask about covenants broadly
print("Turn 1: What financial covenants does this agreement have?\n")
resp1 = qa.ask("What financial covenants does this agreement have?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up referencing the previous answer
print("\nTurn 2: Are there any step-downs in those levels?\n")
resp2 = qa.ask("Are there any step-downs in those levels?", "ribbon_p4")
print_response(resp2)

# Turn 3: Another follow-up
print("\nTurn 3: What happens if the borrower breaches them?\n")
resp3 = qa.ask("What happens if the borrower breaches them?", "ribbon_p4")
print_response(resp3)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What financial covenants does this agreement have?

Answer:
Based on the provided context, this credit agreement has **one financial covenant**:

**Financial Condition Covenant (Section 7.1(a))** - While the specific ratio and threshold are not detailed in the provided excerpts, the agreement includes a financial condition covenant that is tested quarterly based on "four consecutive trailing fiscal quarters of Holdings."

## Key Features of the Financial Covenant:

**Equity Cure Mechanism (Section 7.1(b)):**
- If the Borrower fails to comply with the financial covenant, it can make a cash equity contribution to cure the breach
- The cure contribution must be made within 10 business days after financial statements are due for the relevant quarter
- This "Specified Equity Contribution" gets included in Consolidated Adjusted EBITDA calculations for covenant testing purposes
- During the 10-day cure period, Lenders are not required to fund any new Loans
- There must be at least two

## Step 3: Context Assembly Inspection
Look at the raw prompt being sent to the LLM to verify structure.

In [15]:
from credit_analyzer.generation.prompts import build_context_prompt

# Manually retrieve and build context to inspect
result = retriever.retrieve(
    query="What is the total revolving commitment amount?",
    document_id="ribbon_p4",
    top_k=5,
)

prompt = build_context_prompt(
    chunks=result.chunks,
    definitions=result.injected_definitions,
    history=[],
    question="What is the total revolving commitment amount?",
)

print(f"Prompt length: {len(prompt)} chars")
print(f"Approximate tokens: ~{len(prompt) // 4}")
print("\n" + "=" * 80)
print(prompt)
print("=" * 80)

Prompt length: 5823 chars
Approximate tokens: ~1455

=== CONTEXT FROM CREDIT AGREEMENT ===

--- Source: Defined Terms (Section 1.1, Pages 9-56) ---
“Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment is a sublimit of the Total Revolving
Commitments.

--- Source: Defined Terms (Section 1.1, Pages 9-56) ---
“Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth under the heading “Revolving Commitment” opposite such Lender’s name on
Schedule 1.1A or in the Assignment and Assumption, Incremental Joinder, Extension Amendment or Refinancing Amendment, as
applicable, pursuant to which such Lender became a party hereto, as the same may be changed from time to time pursuant to the terms
hereof (including in connection with assignment

## Step 4: Timing & Token Budget
Verify we're within the 8K context window and check response times.

In [16]:
test_questions = [
    "What is the total revolving commitment amount?",
    "What are the financial covenant test levels?",
    "How much incremental debt can the borrower incur?",
    "Who is the administrative agent?",
    "What are the mandatory prepayment triggers?",
]

print(f"{'Question':<55} {'Time':>6} {'Tokens':>7} {'Conf':>6}")
print("-" * 80)

for q in test_questions:
    qa.clear_history()
    start = time.time()
    resp = qa.ask(q, "ribbon_p4")
    elapsed = time.time() - start
    print(
        f"{q:<55} {elapsed:>5.1f}s {resp.llm_response.tokens_used:>7} {resp.confidence:>6}"
    )

Question                                                  Time  Tokens   Conf
--------------------------------------------------------------------------------
What is the total revolving commitment amount?            5.0s     202   HIGH
What are the financial covenant test levels?              7.9s     261   HIGH
How much incremental debt can the borrower incur?        10.9s     431   HIGH
Who is the administrative agent?                          6.8s     251   HIGH
What are the mandatory prepayment triggers?              12.9s     579   HIGH


## Cleanup

In [17]:
store.delete_collection("ribbon_p4")
print("Cleaned up test collection")

Cleaned up test collection
